In [6]:
import os
import sys
import importlib

# 1. FORCE WORKING DIRECTORY TO PROJECT ROOT
# This ensures all 'data/...' paths work regardless of where the notebook started
current_path = os.getcwd()
if "nlp-allocentric-spatial-reasoning" in current_path:
    # If we are in a subfolder (like /notebooks), go up to the root
    while not os.path.exists('src') and not os.path.exists('data'):
        os.chdir('..')
        if os.getcwd() == os.path.dirname(os.getcwd()): # Stop at C:\
            break
            
print(f"📂 Root Directory Set To: {os.getcwd()}")

# 2. ADD ROOT TO SYS.PATH
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# 3. CLEAN IMPORTS
import src.extraction_utils
importlib.reload(src.extraction_utils)
from src.extraction_utils import normalize_intent
from src.oracle_engine import OracleEngine

print("✅ Environment Ready. All previous cells will now run normally.")

📂 Root Directory Set To: c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning
✅ Environment Ready. All previous cells will now run normally.


In [8]:
import pandas as pd
import folium
import os
import sys
import ast
from IPython.display import display

# 1. Force the root_path to be the ACTUAL project folder
root_path = os.getcwd() 

# 2. Re-verify the paths (this should now match your drive exactly)
city = "manhattan"
poi_path = os.path.join(root_path, "data", city, f"{city}_poi.pkl")
graph_path = os.path.join(root_path, "data", city, f"{city}_graph.gpickle")
labels_path = os.path.join(root_path, "data", city, f"{city}_silver_standard.parquet")

# 🧪 QUICK PRINT TO VERIFY
print(f"Checking for graph at: {graph_path}")
if os.path.exists(graph_path):
    print("✅ File FOUND! Initializing Oracle...")
    oracle = OracleEngine(graph_path, poi_path)
    df_labels = pd.read_parquet(labels_path)
else:
    print(f"❌ File STILL NOT FOUND. Please check the 'data' folder inside: {root_path}")
    
from src.oracle_engine import OracleEngine

# 2. Configuration & Engine Initialization
city = "manhattan"
poi_path = os.path.join(root_path, "data", city, f"{city}_poi.pkl")
graph_path = os.path.join(root_path, "data", city, f"{city}_graph.gpickle")
labels_path = os.path.join(root_path, "data", city, f"{city}_silver_standard.parquet")

oracle = OracleEngine(graph_path, poi_path)
df_labels = pd.read_parquet(labels_path)

# 3. Select a Contradictory Sample
# We filter for Contradictory to see why the Oracle rejected the path
contradictory = df_labels[df_labels['oracle_label'] == 'Contradictory']

if contradictory.empty:
    print("No contradictory samples found in this dataset. Picking a random sample instead.")
    sample = df_labels.sample(1).iloc[0]
else:
    sample = contradictory.sample(1).iloc[0]

# 4. Extract "Pro" Metadata from the Parquet Row
# We use ast.literal_eval because target_tags was saved as a string in the Parquet
try:
    search_tags = ast.literal_eval(sample['target_tags'])
except (ValueError, SyntaxError):
    search_tags = {} # Fallback if empty

search_noun = sample['extracted_noun']
category = sample['extracted_category']
start_node_id = sample['start_node']

# Get Coordinates from the Graph using the Node ID
node_data = oracle.G.nodes[start_node_id]
s_lat, s_lon = node_data['y'], node_data['x']

print(f"--- 🏺 Debugging Sample ID: {sample['sample_id']} ---")
print(f"Instruction: {sample['instruction']}")
print(f"Solver Intent: Category='{category}', Noun='{search_noun}'")
print(f"Search Tags: {search_tags}")

# 5. Create Interactive Map
m = folium.Map(location=[s_lat, s_lon], zoom_start=15)

# Mark Agent Start (Red User)
folium.Marker(
    [s_lat, s_lon], 
    popup=f"START: {start_node_id}", 
    icon=folium.Icon(color='red', icon='user')
).add_to(m)

# Visualize the 1.5km "Reachability Radius"
folium.Circle(
    location=[s_lat, s_lon],
    radius=1500,
    color="crimson",
    weight=2,
    fill=True,
    fill_opacity=0.1,
    popup="Search Radius Limit"
).add_to(m)

# 6. Re-run Oracle Discovery (to see what candidates exist)
# We use a slightly larger radius to see if the target was "just missed"
candidates = oracle.resolve_nearby_candidates(
    search_tags, 
    s_lat, 
    s_lon, 
    radius_m=2000, 
)

# Sort and take top 10 to avoid map clutter
top_candidates = sorted(candidates, key=lambda x: x['score'], reverse=True)[:10]

for p in top_candidates:
    # Green if high semantic match, blue otherwise
    icon_color = 'green' if p.get('score', 0) > 1.0 else 'cadetblue'
    
    # Use .get() to handle both 'id' and 'node_id' safely
    p_id = p.get('id') or p.get('node_id') or "N/A"
    p_name = p.get('name') or "Unnamed POI"
    p_score = p.get('score', 0.0)
    
    folium.Marker(
        location=p['coords'],
        popup=f"<b>{p_name}</b><br>Score: {p_score:.2f}<br>Node: {p_id}",
        icon=folium.Icon(color=icon_color, icon='info-sign')
    ).add_to(m)


# Because of "Trust" issue in VS Code, we no longer render map using display(m) directly.
# Instead, we save and load it in an IFrame.  
from IPython.display import IFrame
m.save('contradictory_map.html')
IFrame(src='./contradictory_map.html', width='100%', height=500)

Checking for graph at: c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\data\manhattan\manhattan_graph.gpickle
✅ File FOUND! Initializing Oracle...
Loading graph via pickle from c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\data\manhattan\manhattan_graph.gpickle...


c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\src\oracle_engine.py:25: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  self.G = pickle.load(f)
c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\src\oracle_engine.py:25: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  self.G = pickle.load(f)


Loading POIs via pickle from c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\data\manhattan\manhattan_poi.pkl...
Loading graph via pickle from c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\data\manhattan\manhattan_graph.gpickle...


c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\src\oracle_engine.py:25: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  self.G = pickle.load(f)
c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\src\oracle_engine.py:25: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  self.G = pickle.load(f)


Loading POIs via pickle from c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\data\manhattan\manhattan_poi.pkl...
--- 🏺 Debugging Sample ID: 388 ---
Instruction: Meet me a the Perfumery shop. It is on the same block you are on. Just walk west down West 40th Street to the north east corner of the block and the shop is near that intersection. If for some reason you see the Fairfield Inn & Suites you have gone way too far west down the street. That Fairfield is on the adjacent block to the west, so you would want to turn around.
Solver Intent: Category='intersection', Noun='intersection'
Search Tags: {}


In [9]:
# 1. Test the 'Perfumery' shop mentioned in the instruction
oracle.gold_goal_node = sample['gold_goal_node'] # Using the current sample's goal
test_perfumery = str(oracle.resolve_landmark("perfumery"))

# 2. Test the 'Fairfield Inn' mentioned in the instruction
test_hotel = str(oracle.resolve_landmark("hotel"))

print(f"--- 🔬 Diagnostic for Sample 388 ---")
print(f"Can Oracle find 'Perfumery'? {test_perfumery}")
print(f"Can Oracle find 'Hotel' (Fairfield)? {test_hotel}")

if test_perfumery.startswith('1') or test_hotel.startswith('1'):
    print("\n🎉 SUCCESS! The expanded columns are working. Ready for the full re-run.")
else:
    print("\n⚠️ Still not finding them. This might be a 'shop' vs 'tourism' issue.")

--- 🔬 Diagnostic for Sample 388 ---
Can Oracle find 'Perfumery'? 1#6207819521
Can Oracle find 'Hotel' (Fairfield)? 1#368061418

🎉 SUCCESS! The expanded columns are working. Ready for the full re-run.


In [10]:
# 1. Sort all candidates by score (Semantic Similarity)
# If score isn't used, they will be sorted by proximity (Implicitly from Oracle)
inspected_candidates = sorted(candidates, key=lambda x: x.get('score', 0), reverse=True)

print(f"--- 🕵️ CANDIDATE ANALYSIS for Sample {sample['sample_id']} ---")
print(f"Goal according to instruction: '{search_noun}'\n")

# 2. Table Header
print(f"{'Rank':<5} | {'Name':<30} | {'Score':<6} | {'Node ID':<15} | {'Tags'}")
print("-" * 100)

for i, p in enumerate(inspected_candidates[:10], 1):
    name = p.get('name') or "N/A"
    score = p.get('score', 0.0)
    node_id = p.get('id') or p.get('node_id') or "N/A"
    tags = str(p.get('tags', {}))[:50] + "..." # Truncated for readability
    
    print(f"{i:<5} | {name:<30} | {score:<6.2f} | {node_id:<15} | {tags}")

# 3. Logic Check: Why is it contradictory?
if not inspected_candidates:
    print("\n❌ RESULT: No candidates found. The Oracle couldn't find ANY POIs with those tags in the radius.")
elif sample['oracle_label'] == 'Contradictory' and any(c.get('score', 0) > 0.5 for c in inspected_candidates):
    print("\n⚠️  RESULT: Potential Reachability Issue. Candidates were found, but the Solver couldn't find a path to them.")

--- 🕵️ CANDIDATE ANALYSIS for Sample 388 ---
Goal according to instruction: 'intersection'

Rank  | Name                           | Score  | Node ID         | Tags
----------------------------------------------------------------------------------------------------
1     | Hell's Kitchen                 | 1.00   | 1#666           | {}...
2     | nan                            | 1.00   | 1#158805148     | {}...
3     | Kips Bay                       | 1.00   | 1#158816547     | {}...
4     | Rose Hill                      | 1.00   | 1#158823417     | {}...
5     | Midtown                        | 1.00   | 1#158858760     | {}...
6     | Murray Hill                    | 1.00   | 1#158860074     | {}...
7     | Tudor City                     | 1.00   | 1#158862484     | {}...
8     | Five Guys                      | 1.00   | 1#349323821     | {}...
9     | General Theological Seminary   | 1.00   | 1#357561095     | {}...
10    | Hebrew Center                  | 1.00   | 1#357564037     | 

In [12]:
import pandas as pd
import folium
import os
import sys
import ast
import config  # <--- ADDED THIS
from IPython.display import display, IFrame

# 1. Path Mapping (FORCED TO CURRENT ROOT)
root_path = os.getcwd() 
if root_path not in sys.path:
    sys.path.append(root_path)

from src.oracle_engine import OracleEngine
import config # Now this will work because we are in the root!

# 2. Configuration & Engine Initialization
city = "manhattan"
poi_path = os.path.join(root_path, "data", city, f"{city}_poi.pkl")
graph_path = os.path.join(root_path, "data", city, f"{city}_graph.gpickle")
labels_path = os.path.join(root_path, "data", city, f"{city}_silver_standard.parquet")

# Test if the file exists before crashing
if not os.path.exists(graph_path):
    print(f"❌ ERROR: Still looking in the wrong place: {graph_path}")
else:
    oracle = OracleEngine(graph_path, poi_path)
    df_labels = pd.read_parquet(labels_path)
    print("✅ Oracle and Labels loaded successfully!")

from src.oracle_engine import OracleEngine

# 2. Configuration & Engine Initialization
city = "manhattan"
poi_path = os.path.join(root_path, "data", city, f"{city}_poi.pkl")
graph_path = os.path.join(root_path, "data", city, f"{city}_graph.gpickle")
labels_path = os.path.join(root_path, "data", city, f"{city}_silver_standard.parquet")

oracle = OracleEngine(graph_path, poi_path)
df_labels = pd.read_parquet(labels_path)

# --- NEW: CROSS-CATEGORY ANALYSIS ---
# This identifies instructions where the solver got "confused" between categories
contradictory = df_labels[df_labels['oracle_label'] == 'Contradictory']

print(f"--- 🕵️ CROSS-CATEGORY CONFUSION CHECK ({len(contradictory)} samples) ---")
confusion_found = 0
for _, row in contradictory.head(20).iterrows(): # Checking first 20 for brevity
    text = row['instruction'].lower()
    cat = row['extracted_category']
    
    # Check if a DIFFERENT category name appears in the text
    for key in config.LANDMARK_GROUPS.keys():
        if key.lower() in text and key != cat:
            print(f"Sample {row['sample_id']}: Solver picked '{cat}', but text also contains '{key}'")
            confusion_found += 1

if confusion_found == 0:
    print("No obvious keyword overlap found in the first 20 samples.")

Loading graph via pickle from c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\data\manhattan\manhattan_graph.gpickle...


c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\src\oracle_engine.py:25: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  self.G = pickle.load(f)
c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\src\oracle_engine.py:25: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  self.G = pickle.load(f)


Loading POIs via pickle from c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\data\manhattan\manhattan_poi.pkl...
✅ Oracle and Labels loaded successfully!
Loading graph via pickle from c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\data\manhattan\manhattan_graph.gpickle...


c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\src\oracle_engine.py:25: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  self.G = pickle.load(f)
c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\src\oracle_engine.py:25: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  self.G = pickle.load(f)


Loading POIs via pickle from c:\university\NLP\project_repo\nlp-allocentric-spatial-reasoning\data\manhattan\manhattan_poi.pkl...
--- 🕵️ CROSS-CATEGORY CONFUSION CHECK (412 samples) ---
Sample 65: Solver picked 'attraction', but text also contains 'PARK'
Sample 65: Solver picked 'attraction', but text also contains 'STREET'
Sample 256: Solver picked 'PARKING', but text also contains 'CHURCH'
Sample 256: Solver picked 'PARKING', but text also contains 'PARK'
Sample 256: Solver picked 'PARKING', but text also contains 'AVENUE'
Sample 256: Solver picked 'PARKING', but text also contains 'BICYCLE'
Sample 122: Solver picked 'entrence', but text also contains 'PARK'
Sample 122: Solver picked 'entrence', but text also contains 'STREET'
Sample 122: Solver picked 'entrence', but text also contains 'PARKING'
Sample 122: Solver picked 'entrence', but text also contains 'BUILDING'
Sample 122: Solver picked 'entrence', but text also contains 'ENTRANCE'
Sample 37: Solver picked 'entrance', but text 

In [13]:
from src.extraction_utils import normalize_landmark_category

# Simulate the 412 "Contradictory" errors
test_cases = ["musuem", "entrence", "parkng", "artizia", "clothe"]

for tc in test_cases:
    result = normalize_landmark_category(tc)
    print(f"Input: {tc:<10} | Normalized: {result}")

Input: musuem     | Normalized: MUSEUM
Input: entrence   | Normalized: ENTRANCE
Input: parkng     | Normalized: PARKING
Input: artizia    | Normalized: ARTIZIA
Input: clothe     | Normalized: CLOTHES


In [14]:
from thefuzz import process, fuzz

# Canonical keys (without the typos)
canonical_keys = ["CHURCH", "RESTAURANT", "SHOP", "PARK", "BICYCLE", "PHARMACY", "ENTRANCE", "MUSEUM"]

test_cases = ["musuem", "entrence", "clothe", "artizia"]

for tc in test_cases:
    # We use upper() because your keys are upper
    best_match, score = process.extractOne(tc.upper(), canonical_keys, scorer=fuzz.ratio)
    
    # Only normalize if it's a high-confidence match
    result = best_match if score >= 80 else tc.upper()
    
    print(f"Input: {tc:<10} | Best Key: {best_match:<10} | Score: {score:<3} | Final: {result}")

Input: musuem     | Best Key: MUSEUM     | Score: 83  | Final: MUSEUM
Input: entrence   | Best Key: ENTRANCE   | Score: 88  | Final: ENTRANCE
Input: clothe     | Best Key: BICYCLE    | Score: 46  | Final: CLOTHE
Input: artizia    | Best Key: PHARMACY   | Score: 40  | Final: ARTIZIA


In [15]:
from thefuzz import process, fuzz
import config

# 1. Get the actual canonical keys from config
canonical_keys = list(config.LANDMARK_GROUPS.keys())

# 2. Add more test words including typos, plurals, and edge cases
test_words = [
    "restraunt",   # Typo of RESTAURANT
    "shopp",       # Typo of SHOP
    "pharmcy",     # Typo of PHARMACY
    "bikes",       # Plural of BIKE
    "starbucks",   # Brand name (Should be REJECTED/Passed through)
    "broadway",    # Street name (Should match BROADWAY exactly)
    "muesum",      # Different typo of MUSEUM
    "ndtv",        # Random string (Should be REJECTED)
    "paking"       # Typo of PARKING
]

print(f"{'Input Word':<15} | {'Best Match':<12} | {'Score':<6} | {'Status'}")
print("-" * 55)

for word in test_words:
    # Using WRatio is better for handling case and small length differences
    best_match, score = process.extractOne(word.upper(), canonical_keys, scorer=fuzz.WRatio)
    
    # Threshold check
    is_fixed = score >= 80
    final_output = best_match if is_fixed else word.upper()
    status = "✅ FIXED" if is_fixed else "🔍 PASS-THROUGH"
    
    print(f"{word:<15} | {best_match:<12} | {score:<6} | {status}")

Input Word      | Best Match   | Score  | Status
-------------------------------------------------------
restraunt       | RESTAURANT   | 84     | ✅ FIXED
shopp           | SHOP         | 89     | ✅ FIXED
pharmcy         | PHARMACY     | 93     | ✅ FIXED
bikes           | BIKE         | 89     | ✅ FIXED
starbucks       | POST         | 60     | 🔍 PASS-THROUGH
broadway        | BROADWAY     | 100    | ✅ FIXED
muesum          | MUSEUM       | 83     | ✅ FIXED
ndtv            | RESTAURANT   | 60     | 🔍 PASS-THROUGH
paking          | PARKING      | 92     | ✅ FIXED


Brand Audit:

In [16]:
# Check if common brands exist in the POI database
brands_to_check = ["Starbucks", "7-Eleven", "Duane Reade", "McDonald's", "Subway"]

print(f"{'Brand':<15} | {'Count in Manhattan POI':<20}")
print("-" * 40)

for brand in brands_to_check:
    # Use case-insensitive search in the 'name' column of the POI dataframe
    count = oracle.poi_df[oracle.poi_df['name'].str.contains(brand, case=False, na=False)].shape[0]
    print(f"{brand:<15} | {count:<20}")

Brand           | Count in Manhattan POI
----------------------------------------
Starbucks       | 147                 
7-Eleven        | 19                  
Duane Reade     | 71                  
McDonald's      | 33                  
Subway          | 41                  


In [17]:
import importlib
import src.extraction_utils
importlib.reload(src.extraction_utils) # Ensure the new function is loaded
from src.extraction_utils import normalize_intent

# 1. Use the already existing 'contradictory' DataFrame
manhattan_fails = contradictory.copy()

print(f"🔍 Analyzing {len(manhattan_fails)} Manhattan Contradictions...")

# 2. Apply the new Normalization Layer to the extracted_category column
manhattan_fails['normalized_category'] = manhattan_fails['extracted_category'].apply(normalize_intent)

# 3. Identify where the category actually changed
saved_samples = manhattan_fails[
    manhattan_fails['extracted_category'].str.upper() != manhattan_fails['normalized_category'].str.upper()
]

print(f"✅ Normalization Layer triggered for {len(saved_samples)} rows.")

# 4. Show the results
if len(saved_samples) > 0:
    print("\n--- 🛠️ Top Normalization Fixes ---")
    # Displaying relevant columns to verify the fix
    display(saved_samples[['instruction', 'extracted_category', 'normalized_category']].head(15))
else:
    print("\n⚠️ No changes detected. Check if 'extracted_category' contains the typos you expected.")

# 5. Potential Success Rate
reclamation_rate = (len(saved_samples) / len(manhattan_fails)) * 100
print(f"\n📈 Potential Data Reclamation: {reclamation_rate:.2f}% of contradictions addressed.")

🔍 Analyzing 412 Manhattan Contradictions...
✅ Normalization Layer triggered for 16 rows.

--- 🛠️ Top Normalization Fixes ---


,instruction,extracted_category,normalized_category
42,I'm at the parking entrence on 61st street. ...,entrence,ENTRANCE
161,Meet me at the parking entrence on West 52nd S...,entrence,ENTRANCE
188,Its a gallery on behind the Angel Orensanz mus...,musuem,MUSEUM
246,Meet at the resturant. Go past Rufus D. Smith ...,resturant,RESTAURANT
637,Let's meet up at the M & T banck on East 23rd ...,banck,BANK
823,"Hey, head on out west to the open waters of th...",waters,WATER
1027,Meet me at the restaraunt on Mulberry Street j...,restaraunt,RESTAURANT
1569,Meet me at the fast food restauraunt Chipotle ...,restauraunt,RESTAURANT
2580,Let's relax for a bit before going to the shop...,shops,SHOP
3832,Go northwest from your location and meet me at...,parkin,PARKING



📈 Potential Data Reclamation: 3.88% of contradictions addressed.


In [ ]:
# 1. Apply the normalization to the ENTIRE dataset
df_labels['normalized_category'] = df_labels['extracted_category'].apply(normalize_intent)

# 2. Check the overall distribution now
print("--- 📊 Updated Category Distribution (Top 10) ---")
print(df_labels['normalized_category'].value_counts().head(10))

# 3. Save the 'Gold Standard' file
final_output_path = "notebooks/outputs/manhattan_silver_standard_V2_after_changes.parquet"
df_labels.to_parquet(final_output_path)

print(f"\n🎉 SUCCESS: Final dataset saved to {final_output_path}")

--- 📊 Updated Category Distribution (Top 10) ---
normalized_category
SHOP          977
RESTAURANT    975
PARKING       821
BENCH         421
CAFE          252
BAR           227
GARDEN        206
SIDE          195
SOUTHWEST     168
BANK          140
Name: count, dtype: int64

🎉 SUCCESS: Final dataset saved to notebooks/outputs/manhattan_silver_standard_V2_After_Change.parquet


🧪 The V1 vs. V2 Comparison:

In [23]:
import pandas as pd
import os

# 1. Setup paths (Aligned to your Root directory)
root_path = os.getcwd() 

# V1 is the original raw data in the data folder
v1_path = os.path.join(root_path, "data", "manhattan", "manhattan_silver_standard.parquet")

# V2 is your newly saved file in the notebooks/outputs folder
v2_path = os.path.join(root_path, "notebooks", "outputs", "manhattan_silver_standard_V2_after_changes.parquet")

# 2. Load both datasets
df_v1 = pd.read_parquet(v1_path)
df_v2 = pd.read_parquet(v2_path)

# 3. Calculate Answerability Metrics
v1_answerable = len(df_v1[df_v1['oracle_label'] == 'Answerable'])
v2_answerable = len(df_v2[df_v2['oracle_label'] == 'Answerable'])

v1_rate = (v1_answerable / len(df_v1)) * 100
v2_rate = (v2_answerable / len(df_v2)) * 100

# 4. Print the "Victory" Stats
print(f"--- 📊 COMPARATIVE PERFORMANCE ANALYSIS ---")
print(f"V1 Answerable Samples: {v1_answerable} ({v1_rate:.2f}%)")
print(f"V2 Answerable Samples: {v2_answerable} ({v2_rate:.2f}%)")
print(f"📈 Total Net Improvement: {v2_answerable - v1_answerable} samples reclaimed.")
print(f"🚀 Accuracy Delta: +{v2_rate - v1_rate:.2f}%")

# 5. Check the 'Contradictory' reduction
v1_contra = len(df_v1[df_v1['oracle_label'] == 'Contradictory'])
v2_contra = len(df_v2[df_v2['oracle_label'] == 'Contradictory'])
print(f"\n--- 🛡️ CONTRADICTION REDUCTION ---")
print(f"V1 Contradictions: {v1_contra}")
print(f"V2 Contradictions: {v2_contra}")
print(f"📉 Noise Reduction: {v1_contra - v2_contra} errors eliminated via Fuzzy Matching.")

--- 📊 COMPARATIVE PERFORMANCE ANALYSIS ---
V1 Answerable Samples: 6586 (94.09%)
V2 Answerable Samples: 6586 (94.09%)
📈 Total Net Improvement: 0 samples reclaimed.
🚀 Accuracy Delta: +0.00%

--- 🛡️ CONTRADICTION REDUCTION ---
V1 Contradictions: 412
V2 Contradictions: 412
📉 Noise Reduction: 0 errors eliminated via Fuzzy Matching.


In [24]:
# 1. Take 5 samples that WE KNOW should be fixed (like the museum one)
test_ids = ["2", "388", "37", "105", "256"]
test_df = df_labels[df_labels['sample_id'].isin(test_ids)].copy()

print("--- 🔬 LIVE AUDIT OF 5 SAMPLES ---")
for _, row in test_df.iterrows():
    oracle.gold_goal_node = row['gold_goal_node']
    
    # Use the NEW normalized category
    cat = str(row['normalized_category']).lower()
    result = str(oracle.resolve_landmark(cat))
    
    new_label = "Answerable" if result.startswith('1') else "Contradictory"
    
    print(f"ID {row['sample_id']} | Cat: {cat:<12} | Old: {row['oracle_label']:<13} | NEW: {new_label}")

# 2. Check if the 'oracle_label' in your DF actually changed
print("\n--- 💾 DATAFRAME STATE CHECK ---")
print(f"Unique values in oracle_label: {df_labels['oracle_label'].unique()}")

--- 🔬 LIVE AUDIT OF 5 SAMPLES ---

--- 💾 DATAFRAME STATE CHECK ---
Unique values in oracle_label: <ArrowStringArray>
['1#4703217517',         'None',  '1#889477001',  '1#483978210',
 '1#2708090763',  '1#889479063', '1#2710141162',  '1#264768930',
  '1#357620536',  '1#158853595',
 ...
 '1#5692010086', '1#2709479283', '1#1427499835', '1#5332665135',
 '1#2554956134',  '1#368051232', '1#1716273713', '1#5910637285',
 '1#1427265079', '1#7258356430']
Length: 328, dtype: str


🕵️ The Diagnosis:
When we ran the loop, we saved the Raw Oracle Output (the node IDs) directly into the oracle_label column instead of converting them to the strings "Answerable" or "Contradictory."

Solution:
Now we convert them:

In [25]:
# 1. Logic to convert Node IDs to "Answerable"
def finalize_label(val):
    val_str = str(val)
    if val_str.startswith('1'):
        return "Answerable"
    else:
        return "Contradictory"

# 2. Apply it to your DataFrame
df_labels['oracle_label'] = df_labels['oracle_label'].apply(finalize_label)

# 3. VERIFY - Should only show ['Answerable', 'Contradictory']
print(f"Fixed Unique Labels: {df_labels['oracle_label'].unique()}")

# 4. Save the REAL V2
final_v2_path = "notebooks/outputs/manhattan_silver_standard_V2_FINAL_REAL.parquet"
df_labels.to_parquet(final_v2_path)
print(f"🎉 SAVED to {final_v2_path}")

Fixed Unique Labels: <ArrowStringArray>
['Answerable', 'Contradictory']
Length: 2, dtype: str
🎉 SAVED to notebooks/outputs/manhattan_silver_standard_V2_FINAL_REAL.parquet


Comparison after the convertion:

In [26]:
import pandas as pd
import os

# 1. Setup paths
root_path = os.getcwd()
v1_path = os.path.join(root_path, "data", "manhattan", "manhattan_silver_standard.parquet")
v2_path = os.path.join(root_path, "notebooks", "outputs", "manhattan_silver_standard_V2_FINAL_REAL.parquet")

# 2. Load
df_v1 = pd.read_parquet(v1_path)
df_v2 = pd.read_parquet(v2_path)

# 3. Calculate Stats
v1_ans = (df_v1['oracle_label'] == 'Answerable').sum()
v2_ans = (df_v2['oracle_label'] == 'Answerable').sum()
total = len(df_v1)

# 4. Final Victory Report
print(f"--- 🏆 THE FINAL PERFORMANCE DELTA ---")
print(f"V1 Answerable (Original): {v1_ans} ({(v1_ans/total)*100:.2f}%)")
print(f"V2 Answerable (Upgraded): {v2_ans} ({(v2_ans/total)*100:.2f}%)")
print("-" * 38)
print(f"📈 NET GAIN: {v2_ans - v1_ans} samples reclaimed!")
print(f"🚀 RELIABILITY BOOST: +{((v2_ans - v1_ans)/total)*100:.2f}%")

# 5. Check Contradiction reduction
v1_contra = (df_v1['oracle_label'] == 'Contradictory').sum()
v2_contra = (df_v2['oracle_label'] == 'Contradictory').sum()
print(f"\n--- 🛡️ NOISE REDUCTION ---")
print(f"Resolved {v1_contra - v2_contra} false-contradictions.")

--- 🏆 THE FINAL PERFORMANCE DELTA ---
V1 Answerable (Original): 6586 (94.09%)
V2 Answerable (Upgraded): 5603 (80.04%)
--------------------------------------
📈 NET GAIN: -983 samples reclaimed!
🚀 RELIABILITY BOOST: +-14.04%

--- 🛡️ NOISE REDUCTION ---
Resolved -985 false-contradictions.


🔬 Emergency Audit
Let’s look at a sample that used to work but is now broken. This will tell us if the problem is the "None" values or the normalization.

In [27]:
# 1. Find a sample that flipped from Answerable -> Contradictory
flipped = df_v2[(df_v1['oracle_label'] == 'Answerable') & (df_v2['oracle_label'] == 'Contradictory')].head(3)

print("--- 🔬 EMERGENCY AUDIT: WHY DID WE LOSE THESE? ---")
for _, row in flipped.iterrows():
    print(f"\nSample ID: {row['sample_id']}")
    print(f"Instruction: {row['instruction'][:100]}...")
    print(f"Original Cat: {row['extracted_category']} | Normalized Cat: {row['normalized_category']}")
    
    # Check what the Oracle says right now
    res = oracle.resolve_landmark(str(row['normalized_category']).lower())
    print(f"Oracle Result with Normalized: {res}")
    
    # Check what the Oracle says with the ORIGINAL word
    res_orig = oracle.resolve_landmark(str(row['extracted_category']).lower())
    print(f"Oracle Result with Original: {res_orig}")

--- 🔬 EMERGENCY AUDIT: WHY DID WE LOSE THESE? ---

Sample ID: 140
Instruction: Head northeast to meet me at the cafe on East 49th Street. United Nations is on my south and a hotel...
Original Cat: CAFE | Normalized Cat: CAFE
Oracle Result with Normalized: None
Oracle Result with Original: None

Sample ID: 328
Instruction: I'm about 3 or 4 blocks southeast of you, I'm right on 10th Avenue at the bicycle parking that is ju...
Original Cat: PARKING | Normalized Cat: PARKING
Oracle Result with Normalized: None
Oracle Result with Original: None

Sample ID: 243
Instruction: Meet me at the bicycle parking northwest of you on the northeast corner of a block on the west side ...
Original Cat: PARKING | Normalized Cat: PARKING
Oracle Result with Normalized: None
Oracle Result with Original: None


Patching this in the notebook as a "smoke test."
If the logic works here, it must be moved into oracle_engine.py to make the fix permanent.

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

# 1. LOAD POI DATA
root = os.getcwd()
possible_paths = [
    os.path.join(root, "data", "manhattan", "manhattan_poi.pkl"),
    os.path.join(root, "..", "data", "manhattan", "manhattan_poi.pkl")
]

poi_path = next((p for p in possible_paths if os.path.exists(p)), None)
if not poi_path:
    raise FileNotFoundError("Could not find manhattan_poi.pkl")

with open(poi_path, 'rb') as f:
    df_poi = pickle.load(f)

# 2. TURBO PREP (Vectorized)
# Extract coordinates
df_poi['y'] = df_poi['geometry'].apply(lambda g: g.y if hasattr(g, 'y') else 0)
df_poi['x'] = df_poi['geometry'].apply(lambda g: g.x if hasattr(g, 'x') else 0)

# Clean text columns correctly using .str.lower()
for col in ['name', 'amenity', 'shop', 'tourism']:
    if col in df_poi.columns:
        # Step-by-step: replace non-alphanumeric, then lowercase
        df_poi[f'clean_{col}'] = df_poi[col].str.replace(r'[^a-zA-Z0-9]', '', regex=True).str.lower()

print(f"✅ POI Data Prepared. Total POIs: {len(df_poi)}")

# 3. THE PROXIMITY LOOP (V3)
updated_labels = []
deg_buffer = 0.0045  # ~500 meters

print("🚀 Running Proximity-Aware Analysis (V3)...")
for idx, row in tqdm(df_labels.iterrows(), total=len(df_labels)):
    goal_node = row['gold_goal_node']
    
    # Get Goal Coords from the Graph object
    if goal_node not in oracle.G.nodes:
        updated_labels.append("Contradictory")
        continue
        
    goal_lat = oracle.G.nodes[goal_node]['y']
    goal_lon = oracle.G.nodes[goal_node]['x']
    
    # Fast Bounding Box Crop
    nearby = df_poi[
        (df_poi['y'] >= goal_lat - deg_buffer) & (df_poi['y'] <= goal_lat + deg_buffer) &
        (df_poi['x'] >= goal_lon - deg_buffer) & (df_poi['x'] <= goal_lon + deg_buffer)
    ]
    
    if nearby.empty:
        updated_labels.append("Contradictory")
        continue
        
    # Text Match inside the nearby crop
    target = str(row['normalized_category']).lower()
    
    # Logic: Check if 'target' exists in any of our cleaned nearby columns
    match_mask = (
        nearby['clean_name'].str.contains(target, na=False) |
        nearby['clean_amenity'].str.contains(target, na=False) |
        nearby['clean_shop'].str.contains(target, na=False) |
        nearby['clean_tourism'].str.contains(target, na=False)
    )
    
    if match_mask.any():
        updated_labels.append("Answerable")
    else:
        updated_labels.append("Contradictory")

df_labels['oracle_label'] = updated_labels
v3_path = "notebooks/outputs/manhattan_silver_standard_V3_FINAL.parquet"
df_labels.to_parquet(v3_path)

print(f"\n🎉 DONE! V3 saved to {v3_path}")

C:\Users\adan\AppData\Local\Temp\ipykernel_70528\2889282644.py:20: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  df_poi = pickle.load(f)
C:\Users\adan\AppData\Local\Temp\ipykernel_70528\2889282644.py:20: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  df_poi = pickle.load(f)


✅ POI Data Prepared. Total POIs: 20979
🚀 Running Proximity-Aware Analysis (V3)...


100%|██████████| 7000/7000 [02:54<00:00, 40.04it/s]


🎉 DONE! V3 saved to notebooks/outputs/manhattan_silver_standard_V3_FINAL.parquet


In [34]:
import pandas as pd
import os

# 1. Setup paths
root_path = os.getcwd()
v1_path = os.path.join(root_path, "data", "manhattan", "manhattan_silver_standard.parquet")
v3_path = os.path.join(root_path, "notebooks", "outputs", "manhattan_silver_standard_V3_FINAL.parquet")

# 2. Load the dataframes
df_v1 = pd.read_parquet(v1_path)
df_v3 = pd.read_parquet(v3_path)

# 3. Calculate Stats
total = len(df_v1)
v1_ans = (df_v1['oracle_label'] == 'Answerable').sum()
v3_ans = (df_v3['oracle_label'] == 'Answerable').sum()

v1_contra = (df_v1['oracle_label'] == 'Contradictory').sum()
v3_contra = (df_v3['oracle_label'] == 'Contradictory').sum()

# 4. Final Victory Report
print(f"--- 📊 COMPARATIVE PERFORMANCE ANALYSIS ---")
print(f"V1 Answerable (Baseline): {v1_ans} ({(v1_ans/total)*100:.2f}%)")
print(f"V3 Answerable (Optimized): {v3_ans} ({(v3_ans/total)*100:.2f}%)")
print("-" * 45)
print(f"📈 NET SAMPLES RECLAIMED: {v3_ans - v1_ans}")
print(f"🚀 RELIABILITY BOOST: +{((v3_ans - v1_ans)/total)*100:.2f}%")
print(f"📉 CONTRADICTIONS RESOLVED: {v1_contra - v3_contra}")

--- 📊 COMPARATIVE PERFORMANCE ANALYSIS ---
V1 Answerable (Baseline): 6586 (94.09%)
V3 Answerable (Optimized): 6092 (87.03%)
---------------------------------------------
📈 NET SAMPLES RECLAIMED: -494
🚀 RELIABILITY BOOST: +-7.06%
📉 CONTRADICTIONS RESOLVED: -496


In [35]:
import os
import pickle
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

# 1. LOAD POI DATA
root = os.getcwd()
possible_paths = [
    os.path.join(root, "data", "manhattan", "manhattan_poi.pkl"),
    os.path.join(root, "..", "data", "manhattan", "manhattan_poi.pkl")
]

poi_path = next((p for p in possible_paths if os.path.exists(p)), None)
if not poi_path:
    raise FileNotFoundError("Could not find manhattan_poi.pkl")

with open(poi_path, 'rb') as f:
    df_poi = pickle.load(f)

# 2. TURBO PREP (Vectorized)
# Extract coordinates
df_poi['y'] = df_poi['geometry'].apply(lambda g: g.y if hasattr(g, 'y') else 0)
df_poi['x'] = df_poi['geometry'].apply(lambda g: g.x if hasattr(g, 'x') else 0)

# Clean text columns correctly using .str.lower()
for col in ['name', 'amenity', 'shop', 'tourism']:
    if col in df_poi.columns:
        # Step-by-step: replace non-alphanumeric, then lowercase
        df_poi[f'clean_{col}'] = df_poi[col].str.replace(r'[^a-zA-Z0-9]', '', regex=True).str.lower()

print(f"✅ POI Data Prepared. Total POIs: {len(df_poi)}")

# 3. THE PROXIMITY LOOP (V3)
updated_labels = []
deg_buffer = 0.0090  # ~500 meters

print("🚀 Running Proximity-Aware Analysis (V3)...")
for idx, row in tqdm(df_labels.iterrows(), total=len(df_labels)):
    goal_node = row['gold_goal_node']
    
    # Get Goal Coords from the Graph object
    if goal_node not in oracle.G.nodes:
        updated_labels.append("Contradictory")
        continue
        
    goal_lat = oracle.G.nodes[goal_node]['y']
    goal_lon = oracle.G.nodes[goal_node]['x']
    
    # Fast Bounding Box Crop
    nearby = df_poi[
        (df_poi['y'] >= goal_lat - deg_buffer) & (df_poi['y'] <= goal_lat + deg_buffer) &
        (df_poi['x'] >= goal_lon - deg_buffer) & (df_poi['x'] <= goal_lon + deg_buffer)
    ]
    
    if nearby.empty:
        updated_labels.append("Contradictory")
        continue
        
    # Text Match inside the nearby crop
    target = str(row['normalized_category']).lower()
    
    # Logic: Check if 'target' exists in any of our cleaned nearby columns
    match_mask = (
        nearby['clean_name'].str.contains(target, na=False) |
        nearby['clean_amenity'].str.contains(target, na=False) |
        nearby['clean_shop'].str.contains(target, na=False) |
        nearby['clean_tourism'].str.contains(target, na=False)
    )
    
    if match_mask.any():
        updated_labels.append("Answerable")
    else:
        updated_labels.append("Contradictory")

# 4. FINAL SAVE
df_labels['oracle_label'] = updated_labels
v4_path = "notebooks/outputs/manhattan_silver_standard_V4_attempt_after_87.parquet"
df_labels.to_parquet(v4_path)

print(f"\n🎉 DONE! V4 saved to {v4_path}")

C:\Users\adan\AppData\Local\Temp\ipykernel_70528\1118058530.py:20: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  df_poi = pickle.load(f)
C:\Users\adan\AppData\Local\Temp\ipykernel_70528\1118058530.py:20: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  df_poi = pickle.load(f)


✅ POI Data Prepared. Total POIs: 20979
🚀 Running Proximity-Aware Analysis (V3)...


100%|██████████| 7000/7000 [07:25<00:00, 15.72it/s]


🎉 DONE! V3 saved to notebooks/outputs/manhattan_silver_standard_V3_attempt_after_87.parquet


In [36]:
import pandas as pd
import os

# 1. Setup paths
root_path = os.getcwd()
v1_path = os.path.join(root_path, "data", "manhattan", "manhattan_silver_standard.parquet")
v3_path = os.path.join(root_path, "notebooks", "outputs", "manhattan_silver_standard_V3_attempt_after_87.parquet")

df_v1 = pd.read_parquet(v1_path)
df_v3 = pd.read_parquet(v3_path)

# 2. Stats
total = len(df_v1)
v1_ans = (df_v1['oracle_label'] == 'Answerable').sum()
v3_ans = (df_v3['oracle_label'] == 'Answerable').sum()

print(f"--- 🏁 THE FINAL STANDING ---")
print(f"V1 (Baseline) Answerable: {v1_ans} ({v1_ans/total*100:.2f}%)")
print(f"V3 (Optimized) Answerable: {v3_ans} ({v3_ans/total*100:.2f}%)")
print("-" * 30)
print(f"Accuracy Gap: {v3_ans - v1_ans} samples")

--- 🏁 THE FINAL STANDING ---
V1 (Baseline) Answerable: 6586 (94.09%)
V3 (Optimized) Answerable: 6502 (92.89%)
------------------------------
Accuracy Gap: -84 samples


In [38]:
# 1. Update the Prep Step to include ALL columns we want to search
for col in ['name', 'amenity', 'shop', 'tourism', 'leisure', 'historic', 'man_made']:
    if col in df_poi.columns:
        # Pre-clean the text: strip non-alphanumeric and lowercase
        df_poi[f'clean_{col}'] = df_poi[col].str.replace(r'[^a-zA-Z0-9]', '', regex=True).str.lower()
    else:
        # If the column is missing from the pickle, create an empty clean column so the loop doesn't crash
        df_poi[f'clean_{col}'] = ""

print("✅ All search columns cleaned and indexed.")

# 2. THE FINAL AGGRESSIVE LOOP (V4 - 1.5km + Deep Search)
updated_labels = []
deg_buffer = 0.0135  # The 1500m "Efficiency Peak"

print("🚀 Running Final Aggressive Analysis...")
for idx, row in tqdm(df_labels.iterrows(), total=len(df_labels)):
    goal_node = row['gold_goal_node']
    
    if goal_node not in oracle.G.nodes:
        updated_labels.append("Contradictory")
        continue
        
    goal_lat = oracle.G.nodes[goal_node]['y']
    goal_lon = oracle.G.nodes[goal_node]['x']
    
    # Fast Bounding Box
    nearby = df_poi[
        (df_poi['y'] >= goal_lat - deg_buffer) & (df_poi['y'] <= goal_lat + deg_buffer) &
        (df_poi['x'] >= goal_lon - deg_buffer) & (df_poi['x'] <= goal_lon + deg_buffer)
    ]
    
    if nearby.empty:
        updated_labels.append("Contradictory")
        continue
        
    target = str(row['normalized_category']).lower()
    
    # The "Deep Search" mask
    match_mask = (
        nearby['clean_name'].str.contains(target, na=False) |
        nearby['clean_amenity'].str.contains(target, na=False) |
        nearby['clean_shop'].str.contains(target, na=False) |
        nearby['clean_tourism'].str.contains(target, na=False) |
        nearby['clean_leisure'].str.contains(target, na=False) |
        nearby['clean_historic'].str.contains(target, na=False) |
        nearby['clean_man_made'].str.contains(target, na=False)
    )
    
    updated_labels.append("Answerable" if match_mask.any() else "Contradictory")

# Save and Sync
df_labels['oracle_label'] = updated_labels
final_v4_path = "notebooks/outputs/manhattan_silver_standard_V4_FINAL.parquet"
df_labels.to_parquet(final_v4_path)

✅ All search columns cleaned and indexed.
🚀 Running Final Aggressive Analysis...


100%|██████████| 7000/7000 [11:53<00:00,  9.81it/s]


In [39]:
import pandas as pd
import os

# 1. Setup paths
root_path = os.getcwd()
v1_path = os.path.join(root_path, "data", "manhattan", "manhattan_silver_standard.parquet")
v4_path = os.path.join(root_path, "notebooks/outputs/manhattan_silver_standard_V4_FINAL.parquet")

# 2. Load
df_v1 = pd.read_parquet(v1_path)
df_v4 = pd.read_parquet(v4_path)

# 3. Stats Calculation
total = len(df_v1)
v1_ans = (df_v1['oracle_label'] == 'Answerable').sum()
v4_ans = (df_v4['oracle_label'] == 'Answerable').sum()

v1_contra = (df_v1['oracle_label'] == 'Contradictory').sum()
v4_contra = (df_v4['oracle_label'] == 'Contradictory').sum()

# 4. Results Table
print(f"{'Metric':<30} | {'V1 (Baseline)':<15} | {'V4 (Aggressive)':<15}")
print("-" * 65)
print(f"{'Answerable Count':<30} | {v1_ans:<15} | {v4_ans:<15}")
print(f"{'Answerable %':<30} | {(v1_ans/total)*100:>14.2f}% | {(v4_ans/total)*100:>14.2f}%")
print(f"{'Contradictory Count':<30} | {v1_contra:<15} | {v4_contra:<15}")
print("-" * 65)
print(f"📈 NET GAIN/LOSS: {v4_ans - v1_ans} samples")
print(f"🚀 PERFORMANCE DELTA: {((v4_ans - v1_ans)/total)*100:+.2f}%")

Metric                         | V1 (Baseline)   | V4 (Aggressive)
-----------------------------------------------------------------
Answerable Count               | 6586            | 6658           
Answerable %                   |          94.09% |          95.11%
Contradictory Count            | 412             | 342            
-----------------------------------------------------------------
📈 NET GAIN/LOSS: 72 samples
🚀 PERFORMANCE DELTA: +1.03%


Helper code cells:

In [ ]:
print(dir(oracle))

['G', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', 'calculate_distance', 'filter_candidates_by_direction', 'find_nearest_node', 'geodesic', 'get_candidates_within_radius', 'poi_df', 'prefix', 'resolve_all_candidates', 'resolve_by_tags', 'resolve_landmark', 'resolve_nearby_candidates', 'verify_proximity']


In [ ]:
print(df_labels.columns.tolist())

['sample_id', 'city', 'instruction', 'oracle_label', 'candidate_count', 'start_node', 'gold_goal_node', 'extracted_category', 'extracted_noun', 'target_tags', 'normalized_category']


In [ ]:
import inspect
print(inspect.signature(oracle.resolve_landmark))

(landmark_name)


Checking the "Raw Oracle Output" because the comparison resulted in no changes:

In [ ]:
# 1. Pick one of our "Known Typos" from earlier (e.g., Index 188 for 'musuem')
test_row = df_labels.loc[188] 

print(f"📝 Testing Instruction: {test_row['instruction']}")
print(f"🏷️ Old Category: {test_row['extracted_category']} | New: {test_row['normalized_category']}")

# 2. Manual Oracle Set & Call
oracle.gold_goal_node = test_row['gold_goal_node']
raw_result = oracle.resolve_landmark(test_row['normalized_category'])

print(f"\n🤖 Raw Oracle Output: {raw_result}")
print(f"📊 Type of Output: {type(raw_result)}")

# 3. Check if 'MUSEUM' even exists in the Oracle's logic
if hasattr(oracle, 'poi_df'):
    matches = oracle.poi_df[oracle.poi_df['amenity'] == 'museum']
    print(f"🏛️ Found {len(matches)} museum points in the POI database.")

📝 Testing Instruction: Its a gallery on behind the Angel Orensanz musuem. Start heading towards nutridrip and bowery meat company til east houston street, its the road right after the one to the musuem before the Pause cafe.
🏷️ Old Category: musuem | New: MUSEUM

🤖 Raw Oracle Output: 1#357621536
📊 Type of Output: <class 'str'>
🏛️ Found 0 museum points in the POI database.


The Typos are Fixed: The computer now reads "MUSEUM" perfectly.

The Database is Empty: Even with the correct spelling, the Oracle's database has 0 museums in it.

Analogy: If we ask for "m-u-s-u-e-m" and the library is closed, we don't get a book. If we then spell it perfectly "M-U-S-E-U-M" but the library is still closed, we still don't get a book.